# Module 4: Neural Networks Basics

Neural networks are function approximators — they learn to map inputs to outputs through layers of linear transformations followed by non-linear activations. This notebook builds a neural network from scratch in PyTorch.

**Why PyTorch over TensorFlow?** PyTorch uses eager execution (runs code line by line like regular Python), making it easier to debug and understand. It's the dominant framework in ML research and increasingly in production.

## 1. Setup and Data

We use the sklearn digits dataset (8x8 pixel images of handwritten digits) — small enough to train quickly on CPU but complex enough to require a real neural network.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import numpy as np
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

digits = load_digits()
X, y = digits.data, digits.target

print(f"Dataset: {X.shape[0]} samples, {X.shape[1]} features (8x8 pixels)")
print(f"Classes: {len(np.unique(y))} digits (0-9)")
print(f"Samples per class: ~{len(y) // 10}")

## 2. Building the Network

**Architecture decisions:**
- **3 hidden layers (128 -> 64 -> 32):** Progressively narrower layers force the network to learn increasingly abstract representations. 128 -> 64 -> 32 -> 10 is a common funnel shape.
- **BatchNorm after each linear layer:** Stabilizes training by normalizing activations, allowing higher learning rates.
- **ReLU activation:** Simple, efficient, avoids vanishing gradient problem. GELU is trendier but ReLU works fine here.
- **Dropout (0.3):** Randomly zeros 30% of neurons during training, preventing co-adaptation and reducing overfitting.

In [ ]:
class MLP(nn.Module):
    def __init__(self, input_dim, hidden_dims, output_dim, dropout=0.3):
        super().__init__()
        layers = []
        prev_dim = input_dim
        for h_dim in hidden_dims:
            layers.extend([
                nn.Linear(prev_dim, h_dim),
                nn.BatchNorm1d(h_dim),
                nn.ReLU(),
                nn.Dropout(dropout),
            ])
            prev_dim = h_dim
        layers.append(nn.Linear(prev_dim, output_dim))
        self.network = nn.Sequential(*layers)

    def forward(self, x):
        return self.network(x)

model = MLP(input_dim=64, hidden_dims=[128, 64, 32], output_dim=10, dropout=0.3)
total_params = sum(p.numel() for p in model.parameters())
print(f"Parameters: {total_params:,}")
print(f"\nArchitecture:\n{model}")

## 3. Training Loop

**Why write our own training loop?** Frameworks like PyTorch Lightning abstract this away, but understanding the raw loop is essential:
1. **Forward pass** — data flows through the network, producing predictions
2. **Loss computation** — CrossEntropyLoss measures how wrong predictions are
3. **Backward pass** — gradients flow backward through the network (backpropagation)
4. **Optimizer step** — Adam updates weights using the gradients
5. **Learning rate scheduling** — StepLR halves the learning rate every 20 epochs to fine-tune

**Why Adam?** It adapts the learning rate per-parameter based on gradient history. Works well out of the box for most problems.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

train_loader = DataLoader(
    TensorDataset(torch.FloatTensor(X_train), torch.LongTensor(y_train)),
    batch_size=64, shuffle=True,
)
test_loader = DataLoader(
    TensorDataset(torch.FloatTensor(X_test), torch.LongTensor(y_test)),
    batch_size=64,
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = MLP(64, [128, 64, 32], 10, dropout=0.3).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=20, gamma=0.5)

print(f"Device: {device}")
print(f"Training for 50 epochs...\n")

best_acc = 0
for epoch in range(50):
    model.train()
    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        optimizer.zero_grad()
        loss = criterion(model(X_batch), y_batch)
        loss.backward()
        optimizer.step()

    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for X_batch, y_batch in test_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            preds = model(X_batch).argmax(dim=1)
            correct += (preds == y_batch).sum().item()
            total += y_batch.size(0)

    test_acc = correct / total
    best_acc = max(best_acc, test_acc)
    scheduler.step()

    if (epoch + 1) % 10 == 0:
        lr = scheduler.get_last_lr()[0]
        print(f"Epoch {epoch+1:3d}: test_acc={test_acc:.4f} (best={best_acc:.4f}) lr={lr:.6f}")

print(f"\nFinal best accuracy: {best_acc:.4f}")

## 4. Activation Function Comparison

**Why does the choice of activation matter?**
- **Sigmoid/Tanh** suffer from vanishing gradients — deep networks train slowly because gradients shrink exponentially through layers
- **ReLU** solves this by having gradient = 1 for positive inputs, but "dead neurons" can occur (gradient = 0 for negative inputs, never recovers)
- **GELU/SiLU** are smooth approximations that avoid dead neurons while maintaining the benefits of ReLU

In [ ]:
class FlexMLP(nn.Module):
    def __init__(self, activation_fn):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(64, 64), activation_fn, nn.Linear(64, 64), activation_fn, nn.Linear(64, 10),
        )
    def forward(self, x):
        return self.net(x)

activations = {
    "ReLU": nn.ReLU(), "LeakyReLU": nn.LeakyReLU(0.1),
    "GELU": nn.GELU(), "Tanh": nn.Tanh(), "Sigmoid": nn.Sigmoid(), "SiLU": nn.SiLU(),
}

X_tr = torch.FloatTensor(X_train)
y_tr = torch.LongTensor(y_train)
X_te = torch.FloatTensor(X_test)
y_te = torch.LongTensor(y_test)

header = f"{'Activation':<15} {'Accuracy':>10}"
print(header)
print("-" * 28)

for name, fn in activations.items():
    m = FlexMLP(fn)
    opt = torch.optim.Adam(m.parameters(), lr=1e-3)
    m.train()
    for _ in range(100):
        opt.zero_grad()
        nn.CrossEntropyLoss()(m(X_tr), y_tr).backward()
        opt.step()
    m.eval()
    with torch.no_grad():
        acc = (m(X_te).argmax(1) == y_te).float().mean().item()
    print(f"{name:<15} {acc:>10.4f}")

## Key Takeaways

1. **Architecture design is about information compression** — funnel from wide to narrow forces abstraction
2. **BatchNorm + Dropout** are the two most important regularization tools in neural networks
3. **Adam optimizer** works well out of the box — start there, switch to SGD + scheduling for fine-tuning
4. **ReLU/GELU are the standard activations** — avoid Sigmoid/Tanh in hidden layers of deep networks
5. **Learning rate is the most important hyperparameter** — schedule it to decay during training